# HEILO Fase 2: Faísca 3.0 (200M) → **Faísca 3.5 (400M)** com conhecimento, código e contas

**O que acontece:** na 1ª sessão a Faísca 3.0 é **esticada de 16 para 32 camadas** (~390 M) sem perder nada (as camadas novas começam "desligadas"). Depois ela treina com uma mistura nova, **sem esquecer o português**:

| Fatia | Peso no fim |
|---|---|
| Português (revisão do corpus da Fase 1) | 60% |
| Conhecimento geral em inglês (FineWeb-Edu, licença ODC-By) | 21% |
| Código Python de licença permissiva (MIT/Apache/BSD) | 12% |
| Contas e raciocínio (gerados aqui, respostas calculadas) | 6% |
| Conversa (só exemplos aprovados por você) | 1% |

O português começa em **80%** e desce para 70% e 60% (um terço da fase cada). A taxa de aprendizado reaquece, fica constante e só cai no fim.

**Régua:** no fim de cada sessão mede R1/R2/R2b e compara com a Faísca 3.0. Se o português (R1) piorar mais de 3%, aparece um **ALERTA** e a versão final não é promovida.

**Dia a dia:** `Executar tudo` com GPU **L4**. Cada sessão prepara no máximo uma parte nova de cada fonte, treina, salva no Drive (`heilo_fase2`) e desliga sozinha. Meta: ~3 bilhões de tokens (~11 sessões de 740 min).

**Antes da 1ª vez:** rode `publicar_heilo.bat` no PC. Precisa da cópia congelada da Fase 1 em `heilo_cerebros/fase1_faisca200m_final.pt` (já existe).

In [ ]:
# 1) Configuração
import time; INICIO = time.time()
REPO = "hugolindo12/HELIO"
BRANCH = "main"
VERSAO = "v3.5"
MINUTOS_SESSAO = 740        # ~12h20 por sessão (com folga para salvar)
TESTAR_AGORA = False        # True = gera uma prévia hoje (o treino continua amanhã)
DESLIGAR_NO_FIM = True      # desliga a máquina do Colab no fim (para de gastar unidades)
import torch, os, sys, json, subprocess, shutil
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative a GPU!"
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print("GPU:", gpu, f"({mem:.0f} GB)")

In [ ]:
# 2) Drive: a Fase 1 fica intacta (só leitura); a Fase 2 tem pasta própria
from google.colab import drive
drive.mount("/content/drive")
PASTA_F1 = "/content/drive/MyDrive/heilo_faisca200"
PASTA = "/content/drive/MyDrive/heilo_fase2"
CEREBROS = "/content/drive/MyDrive/heilo_cerebros"
os.makedirs(PASTA, exist_ok=True)
MARCO_F1 = os.path.join(CEREBROS, "fase1_faisca200m_final.pt")
assert os.path.exists(MARCO_F1), "falta a cópia congelada da Fase 1 em heilo_cerebros"
print("Fase 1 congelada:", MARCO_F1)

In [ ]:
# 3) Código da HEILO (sempre o mais novo)
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
URL = f"https://{TOKEN}@github.com/{REPO}.git"
if not os.path.exists("/content/heilo"):
    subprocess.run(["git", "clone", "-b", BRANCH, URL, "/content/heilo"], check=True)
else:
    subprocess.run("cd /content/heilo && git stash -u -q; git pull -q --rebase", shell=True)
sys.path.insert(0, "/content")
!pip -q install datasets tokenizers
from heilo.training import aurora, faisca_grande, fase2, pretrain, seed_v1, regua
from pathlib import Path
L = faisca_grande.lote_para_gpu(gpu, mem, n_layer=fase2.CONFIG.n_layer)
print("micro-lote", L["micro"], "× acumular", L["acumular"])

## Cérebro e mistura do dia

In [ ]:
# 4) Estica a 3.0 para 400M (só na 1ª vez) + prepara as fontes + aplica os pesos do degrau
tok = faisca_grande.preparar_tokenizer(Path(PASTA_F1))          # o mesmo tokenizer da Fase 1
ck = Path(PASTA) / "pretreino.pt"
estado = fase2.preparar_cerebro(Path(MARCO_F1), ck)
INICIO_FASE, passo_inicial = estado["inicio"], estado["passo"]
PASSOS_TOTAL = INICIO_FASE + fase2.PASSOS_FASE2
print(f"Fase 2: começou no passo {INICIO_FASE:,}; agora no passo {passo_inicial:,} de {PASSOS_TOTAL:,}")

from heilo.training.pipeline import TrainingPipeline
pipe = TrainingPipeline()
split = seed_v1.dividir_sft(pipe)                                 # conversa: só o treino dos aprovados
sobra = MINUTOS_SESSAO - (time.time() - INICIO) / 60
novas = 1 if (sobra > 200 and not TESTAR_AGORA) else 0
fontes = fase2.preparar_fontes(Path(PASTA), tok, split["treino"], novas_por_sessao=novas)
print({f: len(ps) for f, ps in fontes.items()}, "partes prontas por fonte")

# copia tudo para o disco rápido do Colab (ler do Drive direto deixa o treino lento)
def local(p, grupo):
    dst = Path("/content/corpus_local") / grupo / (p.parent.name + "_" + p.name)
    if not (dst / "meta.json").exists():
        shutil.copytree(p, dst, dirs_exist_ok=True)
    return dst
pt = [Path(PASTA_F1) / "wiki"] + pretrain.pastas_corpus(Path(PASTA_F1) / "web")
LOCAIS = {"pt": [local(p, "pt") for p in pt]}
for f, ps in fontes.items():
    LOCAIS[f] = [local(p, f) for p in ps]
d = fase2.degrau(passo_inicial, INICIO_FASE)
W = fase2.pesos(d, fontes)
fase2.aplicar_pesos(LOCAIS, W)
CORPUS = [p for ps in LOCAIS.values() for p in ps]
print(f"degrau {d + 1}/3 → português {1 - sum(W.values()):.0%} | " + " | ".join(f"{k} {v:.1%}" for k, v in W.items()))
print(f"preparação: {(time.time() - INICIO) / 60:.0f} min")

## Treino do dia

In [ ]:
# 5) Treina até completar a sessão (salva no Drive a cada 250 passos)
minutos = MINUTOS_SESSAO - (time.time() - INICIO) / 60 - 25
if TESTAR_AGORA or minutos < 10:
    minutos = 0.01
t0 = time.time()
res = pretrain.pretreinar(fase2.CONFIG, tok, CORPUS, ck, passos=PASSOS_TOTAL, lote=L["micro"], acumular=L["acumular"],
                          lr=1.5e-4, lr_min=1.5e-5, aquecimento=1000, tempo_max_min=minutos,
                          avaliar_cada=250, salvar_cada=250, agenda="wsd", frac_queda=0.2)
modelo = res["modelo"]

In [ ]:
# 6) Progresso e previsão
feitos = res["passo"] - passo_inicial
seg_passo = (time.time() - t0) / max(1, feitos)
prog_arq = Path(PASTA) / "progresso.json"
prog = json.loads(prog_arq.read_text()) if prog_arq.exists() else []
ult = res["hist"][-1] if res["hist"] else {}
if feitos > 0:
    prog.append({"data": time.strftime("%d/%m %H:%M"), "gpu": gpu, "passo": res["passo"], "passos_hoje": feitos,
                 "degrau": d + 1, "perda_val": ult.get("perda_val"), "ppl": ult.get("ppl_val"), "lr": ult.get("lr"),
                 "tokens_fase2_M": round((res["passo"] - INICIO_FASE) * faisca_grande.LOTE_SEQUENCIAS * 1024 / 1e6)})
    prog_arq.write_text(json.dumps(prog, indent=1))
for p in prog:
    print(p)
pct = (res["passo"] - INICIO_FASE) / fase2.PASSOS_FASE2
if feitos > 0:
    faltam = (PASSOS_TOTAL - res["passo"]) * seg_passo / 60 / (MINUTOS_SESSAO - 60)
    print(f"\nPROGRESSO DA FASE 2: {pct:.1%} ({res['passo'] - INICIO_FASE:,}/{fase2.PASSOS_FASE2:,} passos) | faltam ~{faltam:.0f} sessão(ões) de {MINUTOS_SESSAO // 60} h")
modelo.eval()
for inicio in ["O Brasil é", "A fotossíntese é", "Quanto é 6 vezes 8? 6 × 8 =", "def soma(a, b):", "The capital of France is"]:
    ids = [tok.doc] + tok.encode(inicio)
    print("→", inicio + tok.decode(modelo.gerar(ids, max_novos=40, temperatura=0.8, parar_em=(tok.fim, tok.doc),
                                                penalidade=1.2, sem_repetir=4)))
TERMINOU = res["passo"] >= PASSOS_TOTAL
FAZER_VERSAO = TERMINOU or TESTAR_AGORA
NOME_VERSAO = VERSAO if TERMINOU else f"{VERSAO}-p{round(pct * 100)}"
print("\n" + ("FASE 2 COMPLETA: seguindo para o ajuste de conversa e a avaliação." if TERMINOU else
              (f"PRÉVIA {NOME_VERSAO}: ajuste de conversa e avaliação agora; o treino continua amanhã."
               if TESTAR_AGORA else "Sessão salva no Drive. Amanhã é só 'Executar tudo' de novo.")))

In [ ]:
# 6b) Régua fixa + portão anti-esquecimento (compara com a Faísca 3.0)
REGUA = Path(PASTA_F1) / "regua"                 # o mesmo conjunto congelado da Fase 1
PORTAO_OK = True
try:
    r_hoje = regua.medir(modelo, tok, REGUA, rotulo=f"Faísca 400M fase2 passo {res['passo']}")
    hist_r = json.loads((REGUA / "historico.json").read_text())
    base = [h for h in hist_r if h["rotulo"].startswith("Faísca 200M")][-1]
    pt_ = regua.portao(base, r_hoje)
    PORTAO_OK = pt_["aprovado"]
    print(("OK — " if PORTAO_OK else "ALERTA: o português piorou demais — ") + "contra a Faísca 3.0: " + pt_["motivo"])
    if feitos > 0 and prog:
        prog[-1]["regua"] = {k: r_hoje[k] for k in ("r1_ppl", "r2_acerto", "r2b_acerto") if k in r_hoje}
        prog_arq.write_text(json.dumps(prog, indent=1))
except Exception as e:
    print("a régua não mediu hoje (o treino não é afetado):", e)
if TERMINOU:
    marco = Path(CEREBROS) / "fase2_faisca400m_final.pt"
    if not marco.exists():
        shutil.copy2(ck, marco)
        print("cópia congelada guardada em", marco)

## Versão (no fim da Fase 2, ou prévia com TESTAR_AGORA = True)

In [ ]:
# 7) Ajuste de conversa + avaliação + GitHub
if FAZER_VERSAO:
    from heilo.training.ciclos import CiclosSeed
    from heilo.models.seed.gpt import MiniGPTChat
    print(split["manifest"])
    res_sft = pretrain.ajustar_conversa(modelo, split["treino"], split["val"], passos=3000, lote=L["micro"],
                                        lr=3e-5, paciencia=6, pasta_corpus=CORPUS, peso_texto=0.5,
                                        lote_texto=max(2, L["micro"] // 2))
    print("ajuste: guardado o passo", res_sft["passo_escolhido"], "| nota das respostas", res_sft["melhor_nota_geracao"])
    arq = pretrain.exportar(modelo, Path(PASTA) / f"heilo_seed_{NOME_VERSAO}.pt",
                            {"versao": NOME_VERSAO, "linha": "Faísca", "passos_pretreino": res["passo"],
                             "progresso": prog, "sft": res_sft})
    (Path(CEREBROS) / "versoes").mkdir(parents=True, exist_ok=True)
    shutil.copy2(arq, Path(CEREBROS) / "versoes" / Path(arq).name)
    ciclos = CiclosSeed(pipe)
    if not PORTAO_OK:
        print("NÃO vai ser promovida: a régua mostrou que o português piorou mais de 3%.")
    dec = seed_v1.registrar(ciclos, arq, NOME_VERSAO, split["manifest"],
                            {"linha": "Faísca", "tamanho": "400M", "fase": 2, "progresso": prog[-1] if prog else None,
                             "fase2_pct": round(pct * 100, 1), "portao_regua": PORTAO_OK,
                             "sft": {k: res_sft[k] for k in ("perda_val_inicial", "melhor_perda_val",
                                                             "passo_escolhido", "melhor_nota_geracao")}},
                            promover=PORTAO_OK,
                            origem=f"HEILO Faísca 400M: a 3.0 esticada para 32 camadas + Fase 2 ({pct:.0%}: português, "
                                   f"conhecimento geral, código e contas) + ajuste de conversa")
    print("PROMOVIDA" if dec["promovida"] else "NÃO promovida", "—", dec["motivo"])
    chat = MiniGPTChat(arquivo=ciclos.vdir / NOME_VERSAO / "heilo_seed.pt")
    for q in ["oi, tudo bem?", "quem é você?", "qual é a capital de portugal?", "quanto é 6 vezes 8?",
              "o que é uma função em python?", "o que é fotossíntese?", "me fala sobre o Brasil"]:
        print("Você:", q, "\nHEILO:", chat.responder([{"role": "user", "content": q}], temperatura=0.7,
                                                      penalidade=1.2, sem_repetir=4), "\n")
    add = aurora.preparar_para_git(Path("/content/heilo"))
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.email", "heilo@colab"])
    subprocess.run(["git", "-C", "/content/heilo", "config", "user.name", "HEILO (Colab)"])
    subprocess.run(["git", "-C", "/content/heilo", "add", "-f"] + add + ["data/training/versions.json", "data/training/eval"])
    subprocess.run(["git", "-C", "/content/heilo", "commit", "-q", "-m", f"HEILO Faísca {NOME_VERSAO} 400M fase 2 (Colab)"])
    r = subprocess.run("cd /content/heilo && git stash -u -q; git pull --rebase && git push", shell=True)
    print("ENVIADO PARA O GITHUB" if r.returncode == 0 else "ERRO no envio: mande um print desta célula")

## Fim: desliga a máquina (só depois que tudo foi salvo no Drive e no GitHub)

In [ ]:
# 8) Desliga o Colab para não gastar unidades à toa
if DESLIGAR_NO_FIM:
    # espera o Drive terminar de enviar tudo (o pretreino.pt da 400M tem ~4,5 GB)
    print("Esperando o Drive terminar de salvar...")
    from google.colab import drive, runtime
    drive.flush_and_unmount()
    print("Tudo salvo no Drive. Desligando a máquina do Colab para parar de gastar unidades...")
    time.sleep(5)
    runtime.unassign()